# ECG Exploration — MIT-BIH Record 100

**W7D1 — Pre-Stanmore AI for Biomedical Engineering**

This notebook explores the MIT-BIH Arrhythmia Database from PhysioNet. We will:
1. Download record 100
2. Load and inspect the raw ECG signal
3. Load annotations and overlay beat markers
4. Save annotated plot to figures/

## 1. Setup

In [ ]:
import wfdb
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

## 2. Download Record 100

In [ ]:
# Download record 100 from MIT-BIH Arrhythmia Database
wfdb.dl_files('mitdb', dl_dir='data/mitdb', records=['100'])

## 3. Load and Inspect Signal

In [ ]:
# Load first 10 seconds (3600 samples at 360 Hz)
record = wfdb.rdrecord('data/mitdb/100', sampto=3600)

print(f"Sampling frequency: {record.fs} Hz")
print(f"Signal names: {record.sig_name}")
print(f"Units: {record.units}")
print(f"Signal shape: {record.p_signal.shape}")
print(f"Duration: {record.p_signal.shape[0] / record.fs:.1f} seconds")

## 4. Load Annotations

In [ ]:
# Load annotations for the same segment
annotation = wfdb.rdann('data/mitdb/100', 'atr', sampto=3600)

print(f"Number of annotations: {len(annotation.sample)}")
print(f"First 20 symbols: {annotation.symbol[:20]}")

In [ ]:
# Count beat types
beat_counts = Counter(annotation.symbol)
print("\nBeat type counts (first 10 seconds):")
for symbol, count in beat_counts.most_common():
    print(f"  {symbol}: {count}")

## 5. Plot Raw Signal with Beat Markers

In [ ]:
# Create time axis
time = np.arange(record.p_signal.shape[0]) / record.fs

# Get beat samples within our window
beat_samples = annotation.sample[annotation.sample < 3600]
beat_symbols = [annotation.symbol[i] for i, s in enumerate(annotation.sample) if s < 3600]

# Separate normal and other beats
normal_idx = [s for s, sym in zip(beat_samples, beat_symbols) if sym == 'N']
other_idx = [s for s, sym in zip(beat_samples, beat_symbols) if sym != 'N']

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time, record.p_signal[:, 0], label='MLII', color='blue', linewidth=0.5)

# Overlay beat markers
if normal_idx:
    ax.scatter([s/record.fs for s in normal_idx], 
               record.p_signal[normal_idx, 0], 
               color='green', s=20, zorder=5, label='N (normal)')
if other_idx:
    ax.scatter([s/record.fs for s in other_idx], 
               record.p_signal[other_idx, 0], 
               color='red', s=40, zorder=5, label='Other')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (mV)')
ax.set_title('MIT-BIH Record 100 — First 10 seconds (MLII) with Beat Annotations')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Save Figure

In [ ]:
# Replot and save
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time, record.p_signal[:, 0], label='MLII', color='blue', linewidth=0.5)

if normal_idx:
    ax.scatter([s/record.fs for s in normal_idx], 
               record.p_signal[normal_idx, 0], 
               color='green', s=20, zorder=5, label='N (normal)')
if other_idx:
    ax.scatter([s/record.fs for s in other_idx], 
               record.p_signal[other_idx, 0], 
               color='red', s=40, zorder=5, label='Other')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (mV)')
ax.set_title('MIT-BIH Record 100 — First 10 seconds (MLII) with Beat Annotations')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/record100_raw_annotated.png', dpi=150)
print("Saved: figures/record100_raw_annotated.png")

## 7. Summary

### What we learned:
- MIT-BIH records have 2 channels: MLII and V5
- Sampling frequency: 360 Hz
- Annotations mark R-peak locations with beat type symbols
- Most beats in record 100 are normal (N)

### Next steps:
- Day 2: Apply bandpass filter to remove noise
- Day 3: Segment individual heartbeats using R-peaks
- Day 4–5: Build a simple arrhythmia classifier